# PE6201 · A2 · Live Model Battery (Problem A)

**这个 Notebook 只做一件事：用你被分配的那个模型，把整套评估案例跑一遍，得出你自己那一行 pass rate。**

跑完约需 **20–40 分钟**，费用约 **US$0.3**（若你分到 Anthropic haiku 档，约 US$2.8）。用自己的 key，自己付。

---

## 你要做的只有 3 步

1. **先在第 ① 格里填两个东西**（必须在 Run all 之前做完）：
   - **`MODEL`** —— 点开**下拉菜单**，选组长分给你的那个模型（别留在“⚠️ 未选择”）；
   - **`API_KEY`** —— 把**你自己的** OpenRouter key（`sk-or-` 开头）粘进右边的输入框。
2. 菜单 **`Runtime` → `Run all`**（或按 `Ctrl` + `F9`）。之后全部自动，你不用再点任何东西。
3. 跑完后，把最后一块 `=== 抄这一块 ===` 的内容，连同下载下来的 `results.json` 一起发给组长。

> 忘了填就直接 Run all？**不会白跑**：程序会在第 ①、② 格停下，并用红字告诉你回去补哪一步；补好后重新 Run all 即可。
> 第一次用 Colab？不用装任何软件、不用配置环境、不用懂代码。

---

## ⚠️ 三条铁律（违反会让全组数据作废）

| # | 规则 | 为什么 |
|---|---|---|
| 1 | **不要改案例、不要改 prompt** | 全组必须用**完全相同**的案例集和 prompt，否则横向对比不成立 |
| 2 | **只有模型名允许不同** | 模型名是全组之间唯一被允许的变量 |
| 3 | **用你自己的 key** | 共用一个 key，账单和运行记录就分不清是谁跑的 |

## 💰 花费参考（自己的 key，自己付）

| 档位 | 你的模型 | 整套 33 条 约 |
|---|---|---|
| cheap | gpt-4o-mini / Llama 3.1 8B / Qwen 2.5 7B / DeepSeek V3（deepseek-chat） | 48 条 108 runs ≈ US$0.1–0.3 |
| mid | anthropic/claude-haiku-4.5（$1/$5） | ≈ US$1.5–2 |

> 17 Sep 更新：google/gemini-flash-1.5 在 OpenRouter 已无可用端点，M5 改用 deepseek/deepseek-chat（$0.26/$1.03）；Anthropic 请选 claude-haiku-4.5。费用按各模型挂牌价从实测 token 计算（config.MODEL_PRICES），不再按 cheap 档估。

> 课程总额度是 US$10。**千万别选 frontier 档**（一个人一整套 ≈ US$13.78，直接爆额度）。

In [ ]:
#@title ① 只改这一格：选模型 + 填你的 key { display-mode: "form" }
#
#   MODEL   : 点开下拉菜单，选组长分给你的那个模型。
#             （第一项“⚠️ 未选择”是占位，没选的话下一格会挡住你。）
#   API_KEY : 粘贴你自己的 OpenRouter key，sk-or- 开头。
#
#   ⚠️ 关键：先在这一格里选好、填好，再执行 Runtime → Run all。
#            不要先 Run all 再回来改——那样会拿默认模型白跑半小时。
# =====================================================================

MODEL = "⚠️ 未选择（请点开这里选你的模型）" #@param ["⚠️ 未选择（请点开这里选你的模型）", "openai/gpt-4o-mini", "meta-llama/llama-3.1-8b-instruct", "qwen/qwen-2.5-7b-instruct", "google/gemini-2.5-flash-lite", "deepseek/deepseek-chat", "anthropic/claude-haiku-4.5"]
MODEL_OVERRIDE = "" #@param {type:"string"}
API_KEY = "" #@param {type:"string"}

# =====================================================================
import os

MODEL = (MODEL_OVERRIDE.strip() or MODEL.strip())
os.environ["OPENROUTER_API_KEY"] = API_KEY

print("模型 :", MODEL)
if API_KEY.startswith("sk-or-"):
    print("key  : 已填写 (…%s)" % API_KEY[-4:])
else:
    print("key  : !! 还没填，或不是 sk-or- 开头 !!")
    raise RuntimeError(
        "\n\n❌ 第 ① 格的 API_KEY 还空着。\n"
        "   请点开上面那个输入框，把你的 OpenRouter key（sk-or- 开头）粘进去，\n"
        "   然后菜单 Runtime → Run all 重新跑一遍。\n")

if MODEL.startswith("⚠️") or "/" not in MODEL:
    raise RuntimeError(
        "\n\n❌ 第 ① 格的 MODEL 还没选。\n"
        "   请点开上面那个下拉菜单，选组长分给你的那个模型，\n"
        "   然后菜单 Runtime → Run all 重新跑一遍。\n")


In [ ]:
#@title ② 准备代码（自动，不用改。约 20 秒） { display-mode: "form" }
# ---- 前置自检：第 ① 格没填好就停下，别往下跑 ------------------------
if not API_KEY.startswith("sk-or-"):
    raise RuntimeError("❌ 第 ① 格的 API_KEY 还没填（sk-or- 开头）。补好后 Runtime → Run all 重跑。")
if MODEL.startswith("⚠️") or "/" not in MODEL:
    raise RuntimeError("❌ 第 ① 格的 MODEL 还没选。点开下拉选组长分给你的模型，再 Runtime → Run all 重跑。")
# ---------------------------------------------------------------------

import os, re, sys, shutil, subprocess, importlib

REPO     = "https://github.com/didaralmrt-sudo/PE6201_A2_Group5"
WORK     = "/content/PE6201_A2_Group5"
SCAFFOLD = os.path.join(WORK, "A2_scaffold")

# 1) 拉取仓库（公开仓库，无需账号、无需 token）
# 每次都拿仓库 main 的最新版本。原来的 `git pull` 会因为本格改过 config.py
# 而静默失败（check=False），于是复用旧运行时的人一直跑的是旧代码。
if not os.path.isdir(WORK):
    subprocess.run(["git", "clone", "-q", REPO, WORK], check=True)
else:
    subprocess.run(["git", "-C", WORK, "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", WORK, "reset", "-q", "--hard", "origin/main"], check=True)
    subprocess.run(["git", "-C", WORK, "clean", "-qfd", "--exclude=results*.json"], check=False)
print("代码版本 :", subprocess.check_output(["git", "-C", WORK, "log", "-1", "--format=%h %s"], text=True).strip())
os.chdir(SCAFFOLD)
shutil.rmtree("__pycache__", ignore_errors=True)
print("代码位置 :", SCAFFOLD)

# 2) 把 config.py 切到 live，并写入你选的模型（只动这两行，其他一律不碰）
src = open("config.py", encoding="utf-8").read()
src = re.sub(r'^BACKEND\s*=\s*"scripted"', 'BACKEND = "live"', src, count=1, flags=re.M)
src = re.sub(r'^MODEL\s*=\s*".*?"', 'MODEL = "%s"' % MODEL, src, count=1, flags=re.M)
open("config.py", "w", encoding="utf-8").write(src)
shutil.rmtree("__pycache__", ignore_errors=True)   # config.py 自己警告过这个坑

# 3) 心跳：每跑完一个案例打印一行，免得你以为死机了
open("_colab_run.py", "w", encoding="utf-8").write('''
import sys, time
import harness

_orig = harness.run_case
def run_case(cid, **kw):
    t0 = time.time()
    rec = _orig(cid, **kw)
    print("    %-11s turns=%-2s %-10s %6.1fs  $%.4f"
          % (cid, rec.get("turns"), rec.get("stopped_by"),
             time.time() - t0, rec.get("cost_usd", 0)), flush=True)
    return rec

harness.run_case = run_case   # run_set() 在 harness 内部调用它，所以补丁生效
import run_eval
sys.exit(run_eval.main(["run_eval.py"] + sys.argv[1:]))
''')

# 4) 确认配置真的生效了
import config
importlib.reload(config)
print("BACKEND  :", config.BACKEND)
print("MODEL    :", config.MODEL)
print("案例数据 :", config.data_root())


def run(args):
    # 跑 run_eval.py，输出实时刷出来（不然 30 分钟一片安静，你会以为卡死）
    env = dict(os.environ, OPENROUTER_API_KEY=API_KEY)
    p = subprocess.Popen([sys.executable, "-u", "_colab_run.py"] + list(args),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env=env, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    return p.returncode

In [ ]:
#@title ③ 先用 1 个案例试跑（约 1 分钟，≈ US$0.01）—— 验证 key 和模型能用 { display-mode: "form" }
#
#   目的：如果 key 填错了、或这个模型不支持结构化输出，在这里就报错，
#         不至于白等 30 分钟。
#   看到 "DECISION RECORD" 和 "CODE CHECK PASS/FAIL" 就算通过。
# =====================================================================
import json

first = json.load(open("../A2_reference_data/data_A/claims.json",
                       encoding="utf-8"))[0]["claim_id"]
print("试跑案例 :", first)
print("模型     :", MODEL)
print("-" * 68)

code = run([first])

print("-" * 68)
if code == 0:
    print("✅ 试跑通过。继续跑第 ④ 格。")
else:
    print("❌ 试跑失败。把上面的红色报错截图发给组长，先别往下跑。")

In [ ]:
#@title ③b v1 对照模式（可选，仅 M2 需要；默认关闭） { display-mode: "form" }
# =====================================================================
# 只有 M2 需要本格。M2 的作业要求：用「原版 scaffold（v1）」和「改进版（v2）」
# 在【同一个模型】上各跑一遍，对比 pass rate / 成本，证明描述符改进有效。
#
# 本格做什么：
#   把「agent + 描述符」相关文件临时还原成 git 基线提交（d1aad0c，即 vendor
#   原始 scaffold）里的版本，用第①格选的同一个模型跑整套 -> 存成 results_v1.json
#   -> 再还原回你当前的 v2 版本。
#   默认只还原 tools.py / prompt.py（描述符在 tools.DESCRIPTORS，规则在 prompt.RULES），
#   agent.py / guardrails.py 保持 v2（代码层修复不参与对照），【保留 backends.py（案例集）不变】，所以是
#   「同案例、同模型、只换 agent/描述符」的干净对照。
#   勾选 V1_ALSO_REVERT_CASES 则连 backends.py 也还原（案例集回到原版 7 条）。
#
# ⚠️ 前提：你的 v2 改进必须已经写进仓库对应文件（如 tools.py 的 DESCRIPTORS）。
#   如果 tools.py 仍是基线原版（从没改过），那 v1 和 v2 会用同一套描述符，
#   对照看不出差异——这时请先把你的 v2 描述符提交进仓库，再跑本格。
#
# ⚠️ 花费：本格会再跑一整套（20–40 分钟，≈ US$0.27），等于 M2 的 live 跑两倍开销。
#   非 M2 角色请保持 V1_MODE = False，本格直接跳过。
# =====================================================================
V1_MODE = False  #@param {type:"boolean"}
V1_ALSO_REVERT_CASES = False  #@param {type:"boolean"}

import os, shutil, json, datetime, subprocess, importlib

# 还原成 v1 的文件（默认只动 agent/描述符，保留案例集以便对照）
# v1 = the descriptors and the prompt only (D2(b) is a descriptor/prompt
# rewrite). agent.py and guardrails.py stay at v2 so M1's live-loop fixes are
# not reverted and the comparison changes ONE thing.
V1_FILES = ["tools.py", "prompt.py"]
if V1_ALSO_REVERT_CASES:
    V1_FILES.append("backends.py")

# vendor 原始 scaffold 所在的基线提交
BASE = "d1aad0c"   # "scaffold and reference data baseline"

_keep = {}

def _enter_v1():
    for f in V1_FILES:
        cur = os.path.join(SCAFFOLD, f)
        _keep[f] = cur + ".v2keep"
        shutil.copyfile(cur, _keep[f])          # 先存好当前 v2
        try:
            blob = subprocess.check_output(
                ["git", "-C", WORK, "show", "%s:A2_scaffold/%s" % (BASE, f)],
                text=True, stderr=subprocess.STDOUT)
            with open(cur, "w", encoding="utf-8") as fh:
                fh.write(blob)
            print("v1 还原 :", f)
        except Exception as e:
            print("⚠️ 从基线还原 %s 失败：%s —— 跳过（保持 v2）" % (f, str(e).strip()))

def _exit_v1():
    for f in V1_FILES:
        k = _keep.get(f)
        if k and os.path.isfile(k):
            shutil.copyfile(k, os.path.join(SCAFFOLD, f))   # 还原回 v2
            os.remove(k)
    print("已还原回 v2（当前）版本")

if V1_MODE:
    print("===== 开始 v1 对照跑（模型：%s）=====" % MODEL)
    _enter_v1()
    shutil.rmtree(os.path.join(SCAFFOLD, "__pycache__"), ignore_errors=True)
    importlib.reload(config)
    rc = run(["--all"])
    v1_path = os.path.join(SCAFFOLD, "results_v1.json")
    if os.path.isfile(os.path.join(SCAFFOLD, "results.json")):
        shutil.move(os.path.join(SCAFFOLD, "results.json"), v1_path)
    print("EXIT =", rc, "-> 已保存", v1_path)
    try:
        d = json.load(open(v1_path, encoding="utf-8"))
        s = d["summary"]
        print("=" * 62)
        print("=== v1 抄这一块（原版 agent/描述符，%s）===" % MODEL)
        print("mode         : v1 (baseline %s)" % BASE)
        print("model        : %s" % MODEL)
        print("date         : %s" % datetime.date.today().isoformat())
        print("trials       : %s" % s["trials"])
        print("passed       : %s" % s["passed"])
        print("pass_rate    : %.1f%%" % (100.0 * s["pass_rate"]))
        print("median_turns : %s" % s["median_turns"])
        print("cost_usd     : $%.4f" % s["cost_usd"])
        print("=" * 62)
    except Exception as e:
        print("⚠️ 读取 results_v1.json 失败：", e)
    _exit_v1()
    print("v1 对照完成。第④格跑的是 v2（results.json），第⑤格会打印 v2 抄录块；")
    print("本格的 v1 块即你的对照数字。逐条对比请用第⑦格（SRC 选 results_v1.json）。")
else:
    print("V1_MODE = False，跳过 v1 对照。非 M2 角色无需理会本格。")


In [ ]:
#@title ④ 跑完整套（20–40 分钟）—— 中途不要关页面 { display-mode: "form" }
#
#   跑的过程中：每个案例跑完会打印一行，安静是正常的，别关页面。
#   Colab 免费版最长可连续跑 12 小时，足够了。
# =====================================================================
print("开始跑整套评估集。中途安静是正常的，每跑完一个案例会多一行。\n")
code = run(["--all"])
print("\nEXIT =", code)
print("（EXIT = 0 表示正常跑完）")

In [ ]:
#@title ⑤ 抄给组长：这一块 + results.json { display-mode: "form" }
import json, datetime
d = json.load(open("results.json", encoding="utf-8"))
s, res = d["summary"], d["results"]
tin  = sum(r["record"].get("tokens_in",  0) for r in res)
tout = sum(r["record"].get("tokens_out", 0) for r in res)

print("=" * 62)
print("=== 抄这一块，发给组长 ===")
print("model        : %s" % MODEL)
print("date         : %s" % datetime.date.today().isoformat())
print("backend      : %s"
      % ("live" if "BACKEND=live" in d.get("config", "") else "!! 不是 live !!"))
print("trials       : %s" % s["trials"])
print("passed       : %s" % s["passed"])
print("pass_rate    : %.1f%%" % (100.0 * s["pass_rate"]))
print("median_turns : %s" % s["median_turns"])
print("tokens_in    : %s" % tin)
print("tokens_out   : %s" % tout)
print("cost_usd     : $%.4f" % s["cost_usd"])
print("=" * 62)
print()
print("还差一步：跑下面第 ⑥ 格，把 results.json 下载下来一并发给组长。")
if "BACKEND=live" not in d.get("config", ""):
    print()
    print("⚠️⚠️ 这次跑的 BACKEND 不是 live（第 ① 格多半没填好）。")
    print("     这块数据先别发给组长：回第 ① 格把模型和 key 补齐，再 Runtime → Run all 跑一次。")

In [ ]:
#@title ⑥ 下载 results.json（发给组长） { display-mode: "form" }
try:
    from google.colab import files
    files.download("results.json")
    print("已开始下载 results.json —— 把它连同上面那块文字一起发给组长。")
except Exception as e:
    print("自动下载没成功（%r）。" % e)
    print("手动拿：左侧文件夹图标 → content/PE6201_A2_Group5/A2_scaffold/")
    print("        → 右键 results.json → Download")

In [ ]:
#@title ⑦ 逐人提取：只看「你负责的案例」的通过率 { display-mode: "form" }
# =====================================================================
# 第⑤格的抄录块给的是全组聚合 pass_rate。但你的报告通常需要「自己那几条」的
# 通过率。本格读 results.json（或 results_v1.json），按前缀过滤，打印你负责
# 案例的通过率与每条明细。
#
# 用法：
#   PREFIX 填你的角色前缀。M2–M6 的案例 ID 形如 M2-CLM-9101（带 M 前缀），
#           直接填 M2 / M3 / M4 / M5 / M6。
#   M1 的案例是 CLM-xxxx（无 M 前缀），请用 CLM-88 或 CLM-89 这类能圈住你那
#           7 条的前缀；或干脆用 scripted 跑（run_eval.py 默认）拿 13/13 那一档。
#   SRC   选 results.json（v2）或 results_v1.json（v1 对照，仅 M2 有）。
# =====================================================================
PREFIX = "M2"  #@param {type:"string"}
SRC = "results.json"  #@param ["results.json","results_v1.json"]

import os, json
path = os.path.join(SCAFFOLD, SRC)
if not os.path.isfile(path):
    print("❌ 找不到 %s。请先跑完第④格（或 M2 先跑第③b格）。" % path)
else:
    d = json.load(open(path, encoding="utf-8"))
    res = d.get("results", [])
    mine = [r for r in res if str(r.get("case_id", "")).startswith(PREFIX)]
    if not mine:
        print("在 %s 里没找到前缀 %r 的案例。" % (SRC, PREFIX))
        print("确认：1) 你的案例已合并进仓库并跑过；2) PREFIX 填对")
        print("（M2–M6 用 M2..M6，M1 用 CLM-88 之类）。")
    else:
        passed = sum(1 for r in mine if r.get("passed"))
        n = len(mine)
        print("=" * 62)
        print("来源文件 : %s" % SRC)
        print("前缀     : %s" % PREFIX)
        print("案例数   : %d" % n)
        print("通过     : %d" % passed)
        print("通过率   : %.1f%%" % (100.0 * passed / n))
        print("-" * 62)
        for r in sorted(mine, key=lambda x: str(x.get("case_id", ""))):
            rec = r.get("record", {})
            print("  %-14s %s  turns=%-2s  $%.4f"
                  % (r.get("case_id"), "PASS" if r.get("passed") else "FAIL",
                     rec.get("turns"), rec.get("cost_usd", 0)))
        print("=" * 62)
